# Anemia Prediction from Preprocessed Nail Images — RegNet-400MF + Demographics Fusion

This notebook builds the full prediction pipeline shown in your method figures, but **without ITA / skin-tone preprocessing as an input feature**.

Main design:

- Uses **record/patient-level splitting** before image expansion, so images from the same patient cannot appear in train/validation/test at the same time.
- Uses only **training data** to fit scalers, encoders, HGB normalization, and sampling weights.
- Uses **RegNet-Y-400MF** as the image backbone.
- Fuses image features with demographics by **intermediate feature fusion**.
- Trains two heads at once:
  - **Regression:** predict HGB value.
  - **Classification:** predict anemia / non-anemia from HGB threshold.
- Produces performance figures and CSV reports.
- Includes a leakage audit for patient overlap and optional identical-image hash overlap across splits.

Expected input files:

- `FileDirectory.csv`
- `PredictingAnemiaInCo-CJStudyData_DATA_2023-07-12_0653.csv`
- `Race and gender file 070425.csv`
- A folder containing your already-preprocessed nail images. The filenames should still contain the original `stored_name` stem from `FileDirectory.csv`, or match it exactly.


## 0. Setup

Edit `IMAGE_ROOT` to the folder containing your preprocessed nail images.

Examples:

```python
IMAGE_ROOT = Path('/content/drive/MyDrive/anemia/preprocessed_nails')
IMAGE_ROOT = Path('/Users/william/Desktop/preprocessed_png')
```


In [1]:

from pathlib import Path
import os, re, math, json, random, hashlib, warnings
from dataclasses import dataclass
from typing import List, Dict, Tuple, Optional

import numpy as np
import pandas as pd

# ====== EDIT THIS ======
IMAGE_ROOT = Path('/Users/williamtsai/Desktop/NTHU 3.2/special topic/will data/pre_process/brighten_glare/usable_reflection_removed_output/safe_for_training_png')  # <-- change this

FILE_DIRECTORY_CSV = Path('/Users/williamtsai/Desktop/NTHU 3.2/special topic/will data/anemia_prediction/FileDirectory.csv')
LAB_CSV = Path('/Users/williamtsai/Desktop/NTHU 3.2/special topic/will data/anemia_prediction/PredictingAnemiaInCo-CJStudyData_DATA_2023-07-12_0653.csv')
DEMOGRAPHICS_CSV = Path('/Users/williamtsai/Desktop/NTHU 3.2/special topic/will data/anemia_prediction/Race and gender file 070425.csv')

# If running outside this sandbox, change these paths, for example:
# FILE_DIRECTORY_CSV = Path('FileDirectory.csv')
# LAB_CSV = Path('PredictingAnemiaInCo-CJStudyData_DATA_2023-07-12_0653.csv')
# DEMOGRAPHICS_CSV = Path('Race and gender file 070425.csv')

OUT_DIR = Path('/Users/williamtsai/Desktop/NTHU 3.2/special topic/will data/anemia_prediction/anemia_regnet400mf_outputs')
OUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

# Use only original color image fields by default. This excludes eyeimage1bw, eyeimage2bw, etc.
KEEP_FIELD_REGEX = r'^eyeimage[1-5]$'

# Data split. With about 1,047 usable records, test_size=0.10 gives about 101-105 patients.
TEST_SIZE = 0.10
VAL_SIZE = 0.10  # fraction of the whole patient set
HGB_BINS = 8

# Demographic features.
# Default: race/ethnicity are used for subgroup fairness analysis, not as model inputs.
# Set True only if you intentionally want the model to use race/ethnicity as predictors.
INCLUDE_RACE_IN_MODEL = False
INCLUDE_ETHNICITY_IN_MODEL = False

# Leakage checks.
RUN_HASH_LEAKAGE_CHECK = True  # set False if hashing images is too slow

# Training settings.
IMAGE_SIZE = 224
BATCH_SIZE = 24
NUM_WORKERS = 2
EPOCHS = 25
FREEZE_BACKBONE_EPOCHS = 2
LR_HEAD = 1e-3
LR_BACKBONE = 1e-4
WEIGHT_DECAY = 1e-4
REG_LOSS_WEIGHT = 1.0
CLS_LOSS_WEIGHT = 0.5
PATIENCE = 6
MINORITY_CLASS_MULTIPLIER = 2.0

print('Output folder:', OUT_DIR.resolve())


Output folder: /Users/williamtsai/Desktop/NTHU 3.2/special topic/will data/anemia_prediction/anemia_regnet400mf_outputs


## 1. Load and inspect the attached CSV files

This cell checks the real columns and basic shape of the uploaded files before model building.


In [2]:

for p in [FILE_DIRECTORY_CSV, LAB_CSV, DEMOGRAPHICS_CSV]:
    if not p.exists():
        raise FileNotFoundError(f'Missing file: {p}. Please update the path in the setup cell.')

filedir_raw = pd.read_csv(FILE_DIRECTORY_CSV)
lab_raw = pd.read_csv(LAB_CSV)
demo_raw = pd.read_csv(DEMOGRAPHICS_CSV)

print('FileDirectory:', filedir_raw.shape)
display(filedir_raw.head())
print('FileDirectory columns:', filedir_raw.columns.tolist())
print('\nLab / HGB:', lab_raw.shape)
display(lab_raw.head())
print('Lab columns:', lab_raw.columns.tolist())
print('\nRace/Gender:', demo_raw.shape)
display(demo_raw.head())
print('Demographics columns:', demo_raw.columns.tolist())

summary = {
    'file_directory_rows': len(filedir_raw),
    'file_directory_records': int(filedir_raw['record'].nunique()) if 'record' in filedir_raw else None,
    'lab_rows': len(lab_raw),
    'lab_records': int(lab_raw['record_id'].nunique()) if 'record_id' in lab_raw else None,
    'demographics_rows': len(demo_raw),
    'demographics_records': int(demo_raw['Record_ID'].nunique()) if 'Record_ID' in demo_raw else None,
}
print(json.dumps(summary, indent=2))


FileDirectory: (8929, 10)


,project_id,event_id,record,field_name,value,doc_id,stored_name,mime_type,doc_size,stored_date
0,2625,14552,1,eyeimage1,120327,120327,20210604133544_pid2625_CqekU8.jpg,image/jpeg,1658415.0,6/4/2021 13:35
1,2625,14552,1,eyeimage2,120328,120328,20210604133615_pid2625_eEm3I5.jpg,image/jpeg,1414785.0,6/4/2021 13:36
2,2625,14552,1,eyeimage4,120330,120330,20210604133710_pid2625_iDVvJ9.jpg,image/jpeg,1152846.0,6/4/2021 13:37
3,2625,14552,2,eyeimage1,120481,120481,20210607132740_pid2625_7HAkYv.jpg,image/jpeg,1079877.0,6/7/2021 13:27
4,2625,14552,2,eyeimage2,120482,120482,20210607132821_pid2625_PNLUFA.jpg,image/jpeg,1022276.0,6/7/2021 13:28


FileDirectory columns: ['project_id', 'event_id', 'record', 'field_name', 'value', 'doc_id', 'stored_name', 'mime_type', 'doc_size', 'stored_date']

Lab / HGB: (1151, 5)


,record_id,hct,hgb,age,race
0,1,39.1,13.1,21,NaN
1,2,43.6,14.6,33,NaN
2,3,41.7,14.2,48,NaN
3,4,39.1,13.1,31,NaN
4,5,37.2,11.8,79,NaN


Lab columns: ['record_id', 'hct', 'hgb', 'age', 'race']

Race/Gender: (1493, 4)


,Record_ID,sex,race,ethnicity
0,1,F,WHITE,Non-Hispanic
1,2,M,ASIAN - CHINESE,Non-Hispanic
2,3,M,WHITE,Non-Hispanic
3,4,F,WHITE - OTHER EUROPEAN,Non-Hispanic
4,5,F,BLACK/AFRICAN AMERICAN,Non-Hispanic


Demographics columns: ['Record_ID', 'sex', 'race', 'ethnicity']
{
  "file_directory_rows": 8929,
  "file_directory_records": 1053,
  "lab_rows": 1151,
  "lab_records": 1151,
  "demographics_rows": 1493,
  "demographics_records": 1493
}


## 2. Clean HGB, age, sex, race, and ethnicity

Important cleaning choices:

- Converts numeric fields safely.
- Fixes likely swapped `hgb` and `hct` values when `hgb` is impossibly high and `hct` looks like a plausible HGB value.
- Keeps severe anemia values such as HGB around 3.8, but removes impossible values above 20.
- Sets invalid ages to missing, then later imputes age using the **training-set median only**.


In [3]:

def safe_float_series(s: pd.Series) -> pd.Series:
    # Convert mostly-numeric strings to float. Ambiguous values become NaN.
    x = s.astype(str).str.strip().str.replace(',', '.', regex=False)
    # Keep only strings that look like one valid number.
    ok = x.str.fullmatch(r'[+-]?(\d+(\.\d*)?|\.\d+)')
    out = pd.Series(np.nan, index=s.index, dtype='float64')
    out.loc[ok.fillna(False)] = pd.to_numeric(x.loc[ok.fillna(False)], errors='coerce')
    return out

lab = lab_raw.copy()
lab = lab.rename(columns={'record_id': 'record_id'})
lab['record_id'] = pd.to_numeric(lab['record_id'], errors='coerce').astype('Int64')
lab['hgb_num_raw'] = safe_float_series(lab['hgb'])
lab['hct_num_raw'] = safe_float_series(lab['hct'])
lab['age_num_raw'] = safe_float_series(lab['age'])

# Fix likely hgb/hct swap: hgb > 22 is not physiologically plausible for this context,
# while hct between 5 and 20 can plausibly be an HGB value.
swap_mask = (lab['hgb_num_raw'] > 22) & (lab['hct_num_raw'].between(5, 20))
lab['hgb_num'] = lab['hgb_num_raw'].where(~swap_mask, lab['hct_num_raw'])
lab['hct_num'] = lab['hct_num_raw'].where(~swap_mask, lab['hgb_num_raw'])
lab['hgb_hct_swapped'] = swap_mask

# Physiologic filters.
lab.loc[~lab['hgb_num'].between(3.0, 20.0), 'hgb_num'] = np.nan
lab['age_num'] = lab['age_num_raw']
lab.loc[~lab['age_num'].between(0, 120), 'age_num'] = np.nan

# Demographics.
demo = demo_raw.copy().rename(columns={'Record_ID': 'record_id'})
demo['record_id'] = pd.to_numeric(demo['record_id'], errors='coerce').astype('Int64')

def normalize_sex(x):
    if pd.isna(x):
        return 'Not Sure'
    x = str(x).strip().upper()
    if x in ['F', 'FEMALE', 'WOMAN']:
        return 'Female'
    if x in ['M', 'MALE', 'MAN']:
        return 'Male'
    return 'Not Sure'

def clean_text(x, default='Unknown'):
    if pd.isna(x):
        return default
    x = str(x).strip()
    if x == '' or x.lower() in ['nan', 'none', 'unknown/not specified', 'unable to obtain']:
        return default
    return x.upper()

def simplify_race(row):
    race = clean_text(row.get('race', 'Unknown'))
    eth = clean_text(row.get('ethnicity', 'Unknown'))
    if 'HISPANIC' in eth or 'HISPANIC' in race or 'LATINO' in race:
        return 'Hispanic/Latino'
    if 'BLACK' in race or 'AFRICAN' in race or 'HAITIAN' in race or 'CAPE VERDEAN' in race:
        return 'Black'
    if 'ASIAN' in race or 'CHINESE' in race or 'FILIPINO' in race or 'INDIAN' in race:
        return 'Asian'
    if 'WHITE' in race or 'EUROPEAN' in race or 'PORTUGUESE' in race or 'RUSSIAN' in race:
        return 'White'
    if race == 'Unknown':
        return 'Unknown'
    return 'Other'

demo['sex_clean'] = demo['sex'].apply(normalize_sex)
demo['race_clean'] = demo['race'].apply(clean_text)
demo['ethnicity_clean'] = demo['ethnicity'].apply(clean_text)
demo['race_group'] = demo.apply(simplify_race, axis=1)

print('Likely HGB/HCT swapped rows fixed:', int(lab['hgb_hct_swapped'].sum()))
print('Valid HGB rows:', int(lab['hgb_num'].notna().sum()), '/', len(lab))
print('Invalid age rows set to NaN:', int(lab['age_num'].isna().sum()))
print('\nSex counts:')
print(demo['sex_clean'].value_counts(dropna=False))
print('\nRace-group counts:')
print(demo['race_group'].value_counts(dropna=False))

cleaning_report = lab.loc[
    lab['hgb_num'].isna() | lab['age_num'].isna() | lab['hgb_hct_swapped'],
    ['record_id', 'hct', 'hgb', 'age', 'hgb_num_raw', 'hct_num_raw', 'hgb_num', 'age_num', 'hgb_hct_swapped']
]
cleaning_report.to_csv(OUT_DIR / 'cleaning_report_suspicious_rows.csv', index=False)
print('Saved:', OUT_DIR / 'cleaning_report_suspicious_rows.csv')
display(cleaning_report.head(20))


Likely HGB/HCT swapped rows fixed: 2
Valid HGB rows: 1145 / 1151
Invalid age rows set to NaN: 8

Sex counts:
sex_clean
Female      881
Male        610
Not Sure      2
Name: count, dtype: int64

Race-group counts:
race_group
Hispanic/Latino    1469
White                11
Unknown              11
Asian                 2
Name: count, dtype: int64
Saved: /Users/williamtsai/Desktop/NTHU 3.2/special topic/will data/anemia_prediction/anemia_regnet400mf_outputs/cleaning_report_suspicious_rows.csv


,record_id,hct,hgb,age,hgb_num_raw,hct_num_raw,hgb_num,age_num,hgb_hct_swapped
121,122,12.5,39.1,91,39.1,12.5,12.5,91.0,True
122,123,NaN,NaN,56,NaN,NaN,NaN,56.0,False
192,193,32.8,10.6,6e,10.6,32.8,10.6,NaN,False
228,229,46.6,16.0,3433841,16.0,46.6,16.0,NaN,False
258,259,29.7,9.1,2148540,9.1,29.7,9.1,NaN,False
328,329,NaN,NaN,29,NaN,NaN,NaN,29.0,False
408,409,37.3,12.2,NaN,12.2,37.3,12.2,NaN,False
482,483,45.9,14.4,NaN,14.4,45.9,14.4,NaN,False
485,486,Pnd,Pnd,67,NaN,NaN,NaN,67.0,False
502,503,NaN,NaN,72,NaN,NaN,NaN,72.0,False


## 3. Build image-level metadata from FileDirectory + labels + demographics

The model should learn from nail images only. By default this cell keeps `eyeimage1` to `eyeimage5` and excludes `eyeimage1bw`, etc., because black-and-white images can destroy color information that matters for HGB prediction.


In [4]:

filedir = filedir_raw.copy()
required_cols = ['record', 'field_name', 'stored_name', 'mime_type']
missing = [c for c in required_cols if c not in filedir.columns]
if missing:
    raise ValueError(f'FileDirectory.csv is missing required columns: {missing}')

filedir['record_id'] = pd.to_numeric(filedir['record'], errors='coerce').astype('Int64')
filedir['field_name'] = filedir['field_name'].astype(str)
filedir['stored_name'] = filedir['stored_name'].astype(str)
filedir['is_kept_field'] = filedir['field_name'].str.match(KEEP_FIELD_REGEX, na=False)
filedir['is_image_mime'] = filedir['mime_type'].astype(str).str.startswith('image/', na=False)
filedir = filedir[filedir['is_kept_field'] & filedir['is_image_mime']].copy()

meta = (
    filedir
    .merge(lab[['record_id', 'hgb_num', 'age_num', 'hgb_hct_swapped']], on='record_id', how='left')
    .merge(demo[['record_id', 'sex_clean', 'race_clean', 'ethnicity_clean', 'race_group']], on='record_id', how='left')
)
meta['sex_clean'] = meta['sex_clean'].fillna('Not Sure')
meta['race_clean'] = meta['race_clean'].fillna('Unknown')
meta['ethnicity_clean'] = meta['ethnicity_clean'].fillna('Unknown')
meta['race_group'] = meta['race_group'].fillna('Unknown')

# Keep only rows with valid HGB. Age can be missing and will be imputed later from train only.
meta = meta[meta['hgb_num'].notna()].copy()

print('Image metadata rows after filtering:', meta.shape)
print('Unique records:', meta['record_id'].nunique())
print('Fields kept:')
print(meta['field_name'].value_counts())
display(meta.head())

meta.to_csv(OUT_DIR / 'metadata_before_path_resolution.csv', index=False)
print('Saved:', OUT_DIR / 'metadata_before_path_resolution.csv')


Image metadata rows after filtering: (4948, 20)
Unique records: 1047
Fields kept:
field_name
eyeimage1    1041
eyeimage4    1037
eyeimage2    1036
eyeimage5    1004
eyeimage3     830
Name: count, dtype: int64


,project_id,event_id,record,field_name,value,doc_id,stored_name,mime_type,doc_size,stored_date,record_id,is_kept_field,is_image_mime,hgb_num,age_num,hgb_hct_swapped,sex_clean,race_clean,ethnicity_clean,race_group
0,2625,14552,1,eyeimage1,120327,120327,20210604133544_pid2625_CqekU8.jpg,image/jpeg,1658415.0,6/4/2021 13:35,1,True,True,13.1,21.0,False,Female,WHITE,NON-HISPANIC,Hispanic/Latino
1,2625,14552,1,eyeimage2,120328,120328,20210604133615_pid2625_eEm3I5.jpg,image/jpeg,1414785.0,6/4/2021 13:36,1,True,True,13.1,21.0,False,Female,WHITE,NON-HISPANIC,Hispanic/Latino
2,2625,14552,1,eyeimage4,120330,120330,20210604133710_pid2625_iDVvJ9.jpg,image/jpeg,1152846.0,6/4/2021 13:37,1,True,True,13.1,21.0,False,Female,WHITE,NON-HISPANIC,Hispanic/Latino
3,2625,14552,2,eyeimage1,120481,120481,20210607132740_pid2625_7HAkYv.jpg,image/jpeg,1079877.0,6/7/2021 13:27,2,True,True,14.6,33.0,False,Male,ASIAN - CHINESE,NON-HISPANIC,Hispanic/Latino
4,2625,14552,2,eyeimage2,120482,120482,20210607132821_pid2625_PNLUFA.jpg,image/jpeg,1022276.0,6/7/2021 13:28,2,True,True,14.6,33.0,False,Male,ASIAN - CHINESE,NON-HISPANIC,Hispanic/Latino


Saved: /Users/williamtsai/Desktop/NTHU 3.2/special topic/will data/anemia_prediction/anemia_regnet400mf_outputs/metadata_before_path_resolution.csv


## 4. Resolve preprocessed image paths

This cell links each row in `FileDirectory.csv` to your actual preprocessed nail images.

It supports these filename cases:

- Exact same filename, e.g. `20210604133544_pid2625_CqekU8.jpg`
- Same stem with another extension, e.g. `20210604133544_pid2625_CqekU8.png`
- Crop filenames that contain the original stem, e.g. `20210604133544_pid2625_CqekU8_nail_0.png`

If a single original image produced multiple nail crops, all matched crops are included as separate training samples.


In [5]:

IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tif', '.tiff'}

if not IMAGE_ROOT.exists():
    raise FileNotFoundError(
        f'IMAGE_ROOT does not exist: {IMAGE_ROOT}\n'
        'Please edit IMAGE_ROOT in the setup cell to your preprocessed nail-image folder.'
    )

all_files = [p for p in IMAGE_ROOT.rglob('*') if p.is_file() and p.suffix.lower() in IMG_EXTS]
print('Found image files under IMAGE_ROOT:', len(all_files))
if len(all_files) == 0:
    raise FileNotFoundError(f'No image files found under {IMAGE_ROOT}')

# Build fast exact indexes.
name_index: Dict[str, List[Path]] = {}
stem_index: Dict[str, List[Path]] = {}
lower_name_pairs = []
for p in all_files:
    lname = p.name.lower()
    lstem = p.stem.lower()
    name_index.setdefault(lname, []).append(p)
    stem_index.setdefault(lstem, []).append(p)
    lower_name_pairs.append((lname, p))

def resolve_one(stored_name: str) -> List[Path]:
    sn = str(stored_name)
    base = Path(sn).name
    stem = Path(base).stem.lower()
    lname = base.lower()
    found = []
    # 1) exact filename
    found.extend(name_index.get(lname, []))
    # 2) exact stem with any extension
    found.extend(stem_index.get(stem, []))
    # 3) cropped filenames containing original stem
    if not found and stem:
        found.extend([p for ln, p in lower_name_pairs if stem in ln])
    # Remove duplicates while preserving order.
    unique = []
    seen = set()
    for p in found:
        rp = str(p.resolve())
        if rp not in seen:
            seen.add(rp)
            unique.append(p)
    return unique

expanded_rows = []
for _, row in meta.iterrows():
    matches = resolve_one(row['stored_name'])
    for p in matches:
        d = row.to_dict()
        d['image_path'] = str(p)
        d['matched_filename'] = p.name
        expanded_rows.append(d)

image_df = pd.DataFrame(expanded_rows)
print('Rows after resolving image paths:', image_df.shape)
if len(image_df) == 0:
    sample_names = meta['stored_name'].head(10).tolist()
    raise RuntimeError(
        'No FileDirectory stored_name matched files under IMAGE_ROOT.\n'
        f'Sample stored_name values: {sample_names}\n'
        'Your preprocessed filenames may have lost the original stem. Rename them or provide a mapping CSV.'
    )

print('Unique records with matched images:', image_df['record_id'].nunique())
print('Rows per record:')
print(image_df.groupby('record_id').size().describe())

audit_missing = meta[~meta['stored_name'].isin(set(image_df['stored_name']))].copy()
audit_missing.to_csv(OUT_DIR / 'path_resolution_missing_original_images.csv', index=False)
image_df.to_csv(OUT_DIR / 'metadata_with_resolved_image_paths.csv', index=False)
print('Saved:', OUT_DIR / 'metadata_with_resolved_image_paths.csv')
print('Saved:', OUT_DIR / 'path_resolution_missing_original_images.csv')
display(image_df.head())


Found image files under IMAGE_ROOT: 3212
Rows after resolving image paths: (3190, 22)
Unique records with matched images: 820
Rows per record:
count    820.000000
mean       3.890244
std        0.953191
min        1.000000
25%        4.000000
50%        4.000000
75%        4.000000
max        8.000000
dtype: float64
Saved: /Users/williamtsai/Desktop/NTHU 3.2/special topic/will data/anemia_prediction/anemia_regnet400mf_outputs/metadata_with_resolved_image_paths.csv
Saved: /Users/williamtsai/Desktop/NTHU 3.2/special topic/will data/anemia_prediction/anemia_regnet400mf_outputs/path_resolution_missing_original_images.csv


,project_id,event_id,record,field_name,value,doc_id,stored_name,mime_type,doc_size,stored_date,...,is_image_mime,hgb_num,age_num,hgb_hct_swapped,sex_clean,race_clean,ethnicity_clean,race_group,image_path,matched_filename
0,2625,14552,2,eyeimage5,120485,120485,20210607132915_pid2625_eowkBq.jpg,image/jpeg,1507505.0,6/7/2021 13:29,...,True,14.6,33.0,False,Male,ASIAN - CHINESE,NON-HISPANIC,Hispanic/Latino,/Users/williamtsai/Desktop/NTHU 3.2/special to...,20210607132915_pid2625_eowkBq_nail_02.png
1,2625,14552,2,eyeimage5,120485,120485,20210607132915_pid2625_eowkBq.jpg,image/jpeg,1507505.0,6/7/2021 13:29,...,True,14.6,33.0,False,Male,ASIAN - CHINESE,NON-HISPANIC,Hispanic/Latino,/Users/williamtsai/Desktop/NTHU 3.2/special to...,20210607132915_pid2625_eowkBq_nail_03.png
2,2625,14552,2,eyeimage5,120485,120485,20210607132915_pid2625_eowkBq.jpg,image/jpeg,1507505.0,6/7/2021 13:29,...,True,14.6,33.0,False,Male,ASIAN - CHINESE,NON-HISPANIC,Hispanic/Latino,/Users/williamtsai/Desktop/NTHU 3.2/special to...,20210607132915_pid2625_eowkBq_nail_01.png
3,2625,14552,2,eyeimage5,120485,120485,20210607132915_pid2625_eowkBq.jpg,image/jpeg,1507505.0,6/7/2021 13:29,...,True,14.6,33.0,False,Male,ASIAN - CHINESE,NON-HISPANIC,Hispanic/Latino,/Users/williamtsai/Desktop/NTHU 3.2/special to...,20210607132915_pid2625_eowkBq_nail_04.png
4,2625,14552,3,eyeimage5,120494,120494,20210607135137_pid2625_5Ae83W.jpg,image/jpeg,1831450.0,6/7/2021 13:51,...,True,14.2,48.0,False,Male,WHITE,NON-HISPANIC,Hispanic/Latino,/Users/williamtsai/Desktop/NTHU 3.2/special to...,20210607135137_pid2625_5Ae83W_nail_04.png


## 5. Create anemia label and HGB bins

For classification, this notebook uses common adult HGB thresholds:

- Male: anemia if HGB < 13.0 g/dL
- Female: anemia if HGB < 12.0 g/dL
- Not sure / missing sex: anemia if HGB < 12.5 g/dL

For stratification and weighted sampling, HGB is converted into 8 bins at the **record level**.


In [6]:

def anemia_from_hgb_sex(hgb, sex):
    sex = str(sex)
    if sex == 'Male':
        return int(hgb < 13.0)
    if sex == 'Female':
        return int(hgb < 12.0)
    return int(hgb < 12.5)

record_df = (
    image_df
    .sort_values(['record_id', 'image_path'])
    .groupby('record_id', as_index=False)
    .agg({
        'hgb_num': 'first',
        'age_num': 'first',
        'sex_clean': 'first',
        'race_clean': 'first',
        'ethnicity_clean': 'first',
        'race_group': 'first',
        'image_path': 'count',
    })
    .rename(columns={'image_path': 'n_images'})
)
record_df['anemia_label'] = record_df.apply(lambda r: anemia_from_hgb_sex(r['hgb_num'], r['sex_clean']), axis=1)
record_df['hgb_bin'] = pd.qcut(record_df['hgb_num'], q=HGB_BINS, labels=False, duplicates='drop')
record_df['hgb_bin'] = record_df['hgb_bin'].astype(int)

print('Record-level dataset:', record_df.shape)
print('Anemia class counts:')
print(record_df['anemia_label'].value_counts())
print('\nHGB bin counts:')
print(record_df['hgb_bin'].value_counts().sort_index())
print('\nHGB summary:')
display(record_df['hgb_num'].describe())

record_df.to_csv(OUT_DIR / 'record_level_clean_dataset.csv', index=False)
print('Saved:', OUT_DIR / 'record_level_clean_dataset.csv')


Record-level dataset: (820, 10)
Anemia class counts:
anemia_label
1    469
0    351
Name: count, dtype: int64

HGB bin counts:
hgb_bin
0    107
1    104
2     99
3    104
4    107
5     94
6    110
7     95
Name: count, dtype: int64

HGB summary:


count    820.000000
mean      11.551244
std        2.640264
min        4.600000
25%        9.400000
50%       11.900000
75%       13.525000
max       18.900000
Name: hgb_num, dtype: float64

Saved: /Users/williamtsai/Desktop/NTHU 3.2/special topic/will data/anemia_prediction/anemia_regnet400mf_outputs/record_level_clean_dataset.csv


## 6. Patient-level split with leakage checks

This is the core anti-leakage step.

Splitting happens on `record_id`, not on images. After splitting, the split label is copied back to all image rows from the same record.


In [7]:

from sklearn.model_selection import train_test_split

records = record_df.copy()

# Split test first.
try:
    trainval_records, test_records = train_test_split(
        records,
        test_size=TEST_SIZE,
        random_state=SEED,
        stratify=records['hgb_bin']
    )
except ValueError as e:
    print('HGB-bin stratification failed; falling back to anemia stratification:', e)
    trainval_records, test_records = train_test_split(
        records,
        test_size=TEST_SIZE,
        random_state=SEED,
        stratify=records['anemia_label']
    )

# Split validation from remaining records. Convert whole-dataset VAL_SIZE into fraction of trainval.
val_fraction_of_trainval = VAL_SIZE / (1.0 - TEST_SIZE)
try:
    train_records, val_records = train_test_split(
        trainval_records,
        test_size=val_fraction_of_trainval,
        random_state=SEED,
        stratify=trainval_records['hgb_bin']
    )
except ValueError as e:
    print('Validation HGB-bin stratification failed; falling back to anemia stratification:', e)
    train_records, val_records = train_test_split(
        trainval_records,
        test_size=val_fraction_of_trainval,
        random_state=SEED,
        stratify=trainval_records['anemia_label']
    )

split_map = {}
for rid in train_records['record_id']:
    split_map[int(rid)] = 'train'
for rid in val_records['record_id']:
    split_map[int(rid)] = 'val'
for rid in test_records['record_id']:
    split_map[int(rid)] = 'test'

image_df['split'] = image_df['record_id'].astype(int).map(split_map)
if image_df['split'].isna().any():
    raise RuntimeError('Some image rows did not receive a split label.')

# Patient-overlap assertions.
train_ids = set(train_records['record_id'].astype(int))
val_ids = set(val_records['record_id'].astype(int))
test_ids = set(test_records['record_id'].astype(int))
assert train_ids.isdisjoint(val_ids), 'Leakage: train and validation share record IDs.'
assert train_ids.isdisjoint(test_ids), 'Leakage: train and test share record IDs.'
assert val_ids.isdisjoint(test_ids), 'Leakage: validation and test share record IDs.'

split_summary = image_df.groupby('split').agg(
    n_images=('image_path', 'count'),
    n_records=('record_id', 'nunique'),
    mean_hgb=('hgb_num', 'mean'),
    anemia_rate=('hgb_num', lambda s: np.nan),
).reset_index()
# Compute record-level anemia rate per split.
rec_split = record_df.copy()
rec_split['split'] = rec_split['record_id'].astype(int).map(split_map)
rec_summary = rec_split.groupby('split').agg(
    n_records=('record_id', 'count'),
    mean_hgb=('hgb_num', 'mean'),
    anemia_rate=('anemia_label', 'mean'),
    mean_images_per_record=('n_images', 'mean'),
).reset_index()

print('Record-level split summary:')
display(rec_summary)
print('Image-level split counts:')
display(image_df['split'].value_counts())

image_df.to_csv(OUT_DIR / 'metadata_with_splits_no_patient_overlap.csv', index=False)
rec_split.to_csv(OUT_DIR / 'record_level_splits_no_patient_overlap.csv', index=False)
print('Saved:', OUT_DIR / 'metadata_with_splits_no_patient_overlap.csv')
print('Saved:', OUT_DIR / 'record_level_splits_no_patient_overlap.csv')


Record-level split summary:


,split,n_records,mean_hgb,anemia_rate,mean_images_per_record
0,test,82,11.615854,0.585366,3.963415
1,train,656,11.540122,0.573171,3.882622
2,val,82,11.575610,0.548780,3.878049


Image-level split counts:


split
train    2547
test      325
val       318
Name: count, dtype: int64

Saved: /Users/williamtsai/Desktop/NTHU 3.2/special topic/will data/anemia_prediction/anemia_regnet400mf_outputs/metadata_with_splits_no_patient_overlap.csv
Saved: /Users/williamtsai/Desktop/NTHU 3.2/special topic/will data/anemia_prediction/anemia_regnet400mf_outputs/record_level_splits_no_patient_overlap.csv


## 7. Optional identical-image hash leakage check

This catches a stricter leakage case: the same image content copied under different filenames and assigned to different splits.

If any identical image hash appears in more than one split, the notebook stops and writes a report.


In [8]:

def file_md5(path, chunk_size=1024 * 1024):
    h = hashlib.md5()
    with open(path, 'rb') as f:
        while True:
            b = f.read(chunk_size)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

if RUN_HASH_LEAKAGE_CHECK:
    hash_rows = []
    for i, row in image_df.iterrows():
        if i % 1000 == 0:
            print(f'Hashing {i}/{len(image_df)}')
        p = row['image_path']
        hash_rows.append({
            'image_path': p,
            'record_id': int(row['record_id']),
            'split': row['split'],
            'md5': file_md5(p),
        })
    hash_df = pd.DataFrame(hash_rows)
    hash_df.to_csv(OUT_DIR / 'image_md5_hashes_by_split.csv', index=False)

    leakage_hashes = []
    for md5, g in hash_df.groupby('md5'):
        if g['split'].nunique() > 1:
            leakage_hashes.append(g)
    if leakage_hashes:
        leakage_df = pd.concat(leakage_hashes, ignore_index=True)
        leakage_df.to_csv(OUT_DIR / 'LEAKAGE_identical_image_hash_across_splits.csv', index=False)
        display(leakage_df.head(30))
        raise AssertionError(
            'Identical image content appears across different splits. '
            f'Report saved to {OUT_DIR / "LEAKAGE_identical_image_hash_across_splits.csv"}'
        )
    else:
        print('PASS: no identical image hash appears across different splits.')
else:
    print('Skipped hash leakage check. Patient-level split check was still performed.')


Hashing 0/3190
Hashing 1000/3190
Hashing 2000/3190
Hashing 3000/3190
PASS: no identical image hash appears across different splits.


## 8. Fit demographics preprocessing using training records only

No ITA skin tone feature is used here.

Default model inputs:

- Age z-score, fitted on training records only and rounded to 1 decimal.
- Sex one-hot encoding: `[Male, Not Sure, Female]` style categories are learned from the training records.
- Race and ethnicity are available for fairness reports. They are not used as model inputs unless enabled in the setup cell.


In [9]:

from sklearn.preprocessing import OneHotEncoder
import joblib

train_rec = rec_split[rec_split['split'] == 'train'].copy()

age_median = float(train_rec['age_num'].median())
age_mean = float(train_rec['age_num'].fillna(age_median).mean())
age_std = float(train_rec['age_num'].fillna(age_median).std(ddof=0))
if age_std == 0 or np.isnan(age_std):
    age_std = 1.0

cat_cols = ['sex_clean']
if INCLUDE_RACE_IN_MODEL:
    cat_cols.append('race_group')
if INCLUDE_ETHNICITY_IN_MODEL:
    cat_cols.append('ethnicity_clean')

try:
    ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
except TypeError:
    ohe = OneHotEncoder(handle_unknown='ignore', sparse=False)

ohe.fit(train_rec[cat_cols].fillna('Unknown'))

def add_demo_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    age = out['age_num'].fillna(age_median).astype(float)
    out['age_z'] = ((age - age_mean) / age_std).round(1)
    cat = ohe.transform(out[cat_cols].fillna('Unknown'))
    try:
        cat_names = ohe.get_feature_names_out(cat_cols).tolist()
    except Exception:
        cat_names = [f'cat_{i}' for i in range(cat.shape[1])]
    cat_df = pd.DataFrame(cat, columns=cat_names, index=out.index)
    out = pd.concat([out, cat_df], axis=1)
    return out, ['age_z'] + cat_names

image_df_fe, demo_feature_cols = add_demo_features(image_df)
rec_split_fe, _ = add_demo_features(rec_split)

# Normalize regression target using training records only, then apply to image rows.
hgb_mean = float(train_rec['hgb_num'].mean())
hgb_std = float(train_rec['hgb_num'].std(ddof=0))
if hgb_std == 0 or np.isnan(hgb_std):
    hgb_std = 1.0
image_df_fe['hgb_z'] = (image_df_fe['hgb_num'] - hgb_mean) / hgb_std
image_df_fe['anemia_label'] = image_df_fe.apply(lambda r: anemia_from_hgb_sex(r['hgb_num'], r['sex_clean']), axis=1)

preprocess_info = {
    'age_median_train': age_median,
    'age_mean_train': age_mean,
    'age_std_train': age_std,
    'cat_cols': cat_cols,
    'demo_feature_cols': demo_feature_cols,
    'hgb_mean_train': hgb_mean,
    'hgb_std_train': hgb_std,
    'include_race_in_model': INCLUDE_RACE_IN_MODEL,
    'include_ethnicity_in_model': INCLUDE_ETHNICITY_IN_MODEL,
}
joblib.dump({'ohe': ohe, 'preprocess_info': preprocess_info}, OUT_DIR / 'demographics_preprocessor_train_only.joblib')
image_df_fe.to_csv(OUT_DIR / 'model_ready_image_metadata.csv', index=False)
rec_split_fe.to_csv(OUT_DIR / 'model_ready_record_metadata.csv', index=False)

print('Demographic feature columns:', demo_feature_cols)
print('Demo feature dimension:', len(demo_feature_cols))
print('HGB mean/std fitted on TRAIN records only:', hgb_mean, hgb_std)
print('Saved:', OUT_DIR / 'demographics_preprocessor_train_only.joblib')
display(image_df_fe[['record_id', 'split', 'hgb_num', 'hgb_z', 'age_num', 'age_z', 'sex_clean', 'race_group'] + demo_feature_cols].head())


Demographic feature columns: ['age_z', 'sex_clean_Female', 'sex_clean_Male', 'sex_clean_Not Sure']
Demo feature dimension: 4
HGB mean/std fitted on TRAIN records only: 11.540121951219511 2.640107568439736
Saved: /Users/williamtsai/Desktop/NTHU 3.2/special topic/will data/anemia_prediction/anemia_regnet400mf_outputs/demographics_preprocessor_train_only.joblib


,record_id,split,hgb_num,hgb_z,age_num,age_z,sex_clean,race_group,age_z,sex_clean_Female,sex_clean_Male,sex_clean_Not Sure
0,2,train,14.6,1.158997,33.0,-1.5,Male,Hispanic/Latino,-1.5,0.0,1.0,0.0
1,2,train,14.6,1.158997,33.0,-1.5,Male,Hispanic/Latino,-1.5,0.0,1.0,0.0
2,2,train,14.6,1.158997,33.0,-1.5,Male,Hispanic/Latino,-1.5,0.0,1.0,0.0
3,2,train,14.6,1.158997,33.0,-1.5,Male,Hispanic/Latino,-1.5,0.0,1.0,0.0
4,3,train,14.2,1.007489,48.0,-0.7,Male,Hispanic/Latino,-0.7,0.0,1.0,0.0


## 9. Create PyTorch datasets and dataloaders

Only the training transform uses augmentation.

Validation and test transforms only resize and normalize.


In [10]:

try:
    import torch
    import torch.nn as nn
    from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
    from PIL import Image
    import torchvision
    import torchvision.transforms as T
except Exception as e:
    raise ImportError(
        'This notebook needs torch, torchvision, pillow. Install them first, e.g.\n'
        'pip install torch torchvision pillow scikit-learn matplotlib pandas numpy joblib\n'
        f'Original import error: {e}'
    )

# Reproducibility.
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = T.Compose([
    T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomVerticalFlip(p=0.5),
    T.RandomRotation(degrees=15),
    T.RandomAffine(degrees=0, translate=(0.08, 0.08)),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

eval_transform = T.Compose([
    T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

class NailAnemiaDataset(Dataset):
    def __init__(self, df, demo_cols, transform=None):
        self.df = df.reset_index(drop=True).copy()
        self.demo_cols = demo_cols
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['image_path']).convert('RGB')
        if self.transform:
            img = self.transform(img)
        demo = torch.tensor(row[self.demo_cols].astype(float).values, dtype=torch.float32)
        y_reg = torch.tensor([float(row['hgb_z'])], dtype=torch.float32)
        y_cls = torch.tensor([float(row['anemia_label'])], dtype=torch.float32)
        meta_item = {
            'record_id': int(row['record_id']),
            'image_path': row['image_path'],
            'hgb_true': float(row['hgb_num']),
            'sex_clean': row['sex_clean'],
            'race_group': row['race_group'],
            'ethnicity_clean': row['ethnicity_clean'],
            'split': row['split'],
        }
        return img, demo, y_reg, y_cls, meta_item

train_df = image_df_fe[image_df_fe['split'] == 'train'].copy()
val_df = image_df_fe[image_df_fe['split'] == 'val'].copy()
test_df = image_df_fe[image_df_fe['split'] == 'test'].copy()

train_ds = NailAnemiaDataset(train_df, demo_feature_cols, train_transform)
val_ds = NailAnemiaDataset(val_df, demo_feature_cols, eval_transform)
test_ds = NailAnemiaDataset(test_df, demo_feature_cols, eval_transform)

# Weighted sampling: inverse HGB-bin frequency, with minority anemia class multiplied.
train_record_bins = rec_split[['record_id', 'hgb_bin']].copy()
train_df = train_df.merge(train_record_bins, on='record_id', how='left', suffixes=('', '_rec'))
bin_counts = train_df['hgb_bin'].value_counts().to_dict()
class_counts = train_df['anemia_label'].value_counts().to_dict()
minority_class = min(class_counts, key=class_counts.get) if len(class_counts) == 2 else None
sample_weights = []
for _, r in train_df.iterrows():
    w = 1.0 / bin_counts.get(r['hgb_bin'], 1)
    if minority_class is not None and int(r['anemia_label']) == int(minority_class):
        w *= MINORITY_CLASS_MULTIPLIER
    sample_weights.append(w)
sample_weights = torch.DoubleTensor(sample_weights)
sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print('Train/val/test image rows:', len(train_ds), len(val_ds), len(test_ds))
print('Train/val/test patient records:', train_df['record_id'].nunique(), val_df['record_id'].nunique(), test_df['record_id'].nunique())


Device: cpu
Train/val/test image rows: 2547 318 325
Train/val/test patient records: 656 82 82


## 10. Model: RegNet-Y-400MF + demographics intermediate fusion

The model follows the architecture idea in your screenshot:

`input nail image -> RegNet image features`

`demographics vector -> demographics MLP`

`concatenate image features + demographics features -> fusion layer -> regression head + classification head`


In [11]:

class RegNet400MFFusion(nn.Module):
    def __init__(self, demo_dim: int, pretrained: bool = True):
        super().__init__()
        # Torchvision changed the pretrained API, so support both old and new styles.
        try:
            from torchvision.models import RegNet_Y_400MF_Weights
            weights = RegNet_Y_400MF_Weights.DEFAULT if pretrained else None
            backbone = torchvision.models.regnet_y_400mf(weights=weights)
        except Exception:
            backbone = torchvision.models.regnet_y_400mf(pretrained=pretrained)

        in_features = backbone.fc.in_features
        backbone.fc = nn.Identity()
        self.backbone = backbone

        self.demo_mlp = nn.Sequential(
            nn.Linear(demo_dim, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(inplace=True),
            nn.Dropout(0.20),
            nn.Linear(32, 32),
            nn.ReLU(inplace=True),
        )

        self.fusion = nn.Sequential(
            nn.Linear(in_features + 32, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.30),
        )
        self.reg_head = nn.Linear(256, 1)
        self.cls_head = nn.Linear(256, 1)

    def forward(self, image, demo):
        img_feat = self.backbone(image)
        demo_feat = self.demo_mlp(demo)
        fused = torch.cat([img_feat, demo_feat], dim=1)
        z = self.fusion(fused)
        hgb_z = self.reg_head(z)
        anemia_logit = self.cls_head(z)
        return hgb_z, anemia_logit


def set_backbone_trainable(model, trainable: bool):
    for p in model.backbone.parameters():
        p.requires_grad = trainable

model = RegNet400MFFusion(demo_dim=len(demo_feature_cols), pretrained=True).to(device)
set_backbone_trainable(model, False)  # warmup/freeze phase

print(model.__class__.__name__)
print('Total parameters:', sum(p.numel() for p in model.parameters()))
print('Trainable parameters during freeze phase:', sum(p.numel() for p in model.parameters() if p.requires_grad))


RegNet400MFFusion
Total parameters: 4026026
Trainable parameters during freeze phase: 122882


## 11. Training and evaluation functions

Record-level metrics are computed by averaging predictions from all nail images belonging to the same patient/record.


In [12]:

from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, balanced_accuracy_score, precision_recall_fscore_support,
    confusion_matrix, roc_auc_score, average_precision_score, roc_curve, precision_recall_curve
)
from scipy.stats import pearsonr, spearmanr
import matplotlib.pyplot as plt

# Classification positive weight from training image rows.
pos = float((train_df['anemia_label'] == 1).sum())
neg = float((train_df['anemia_label'] == 0).sum())
pos_weight_value = neg / max(pos, 1.0)
pos_weight = torch.tensor([pos_weight_value], dtype=torch.float32, device=device)
print('BCE pos_weight:', pos_weight_value)

reg_loss_fn = nn.SmoothL1Loss()
cls_loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

# Parameter groups allow a lower learning rate for the pretrained backbone after unfreezing.
def make_optimizer(model, backbone_trainable=False):
    head_params = []
    backbone_params = []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if name.startswith('backbone.'):
            backbone_params.append(p)
        else:
            head_params.append(p)
    groups = []
    if head_params:
        groups.append({'params': head_params, 'lr': LR_HEAD})
    if backbone_params:
        groups.append({'params': backbone_params, 'lr': LR_BACKBONE})
    return torch.optim.AdamW(groups, weight_decay=WEIGHT_DECAY)

optimizer = make_optimizer(model, backbone_trainable=False)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=2, factor=0.5)


def train_one_epoch(model, loader, optimizer):
    model.train()
    total_loss = total_reg = total_cls = 0.0
    n = 0
    for images, demos, y_reg, y_cls, meta_item in loader:
        images = images.to(device, non_blocking=True)
        demos = demos.to(device, non_blocking=True)
        y_reg = y_reg.to(device, non_blocking=True)
        y_cls = y_cls.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        pred_reg, pred_logit = model(images, demos)
        loss_reg = reg_loss_fn(pred_reg, y_reg)
        loss_cls = cls_loss_fn(pred_logit, y_cls)
        loss = REG_LOSS_WEIGHT * loss_reg + CLS_LOSS_WEIGHT * loss_cls
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        bs = images.size(0)
        total_loss += float(loss.item()) * bs
        total_reg += float(loss_reg.item()) * bs
        total_cls += float(loss_cls.item()) * bs
        n += bs
    return {'loss': total_loss / n, 'reg_loss': total_reg / n, 'cls_loss': total_cls / n}


def collect_predictions(model, loader):
    model.eval()
    rows = []
    total_loss = total_reg = total_cls = 0.0
    n = 0
    with torch.no_grad():
        for images, demos, y_reg, y_cls, meta_item in loader:
            images = images.to(device, non_blocking=True)
            demos = demos.to(device, non_blocking=True)
            y_reg = y_reg.to(device, non_blocking=True)
            y_cls = y_cls.to(device, non_blocking=True)
            pred_reg_z, pred_logit = model(images, demos)
            loss_reg = reg_loss_fn(pred_reg_z, y_reg)
            loss_cls = cls_loss_fn(pred_logit, y_cls)
            loss = REG_LOSS_WEIGHT * loss_reg + CLS_LOSS_WEIGHT * loss_cls

            pred_hgb = pred_reg_z.detach().cpu().numpy().reshape(-1) * hgb_std + hgb_mean
            prob = torch.sigmoid(pred_logit).detach().cpu().numpy().reshape(-1)
            true_hgb = y_reg.detach().cpu().numpy().reshape(-1) * hgb_std + hgb_mean
            true_cls = y_cls.detach().cpu().numpy().reshape(-1).astype(int)

            bs = images.size(0)
            total_loss += float(loss.item()) * bs
            total_reg += float(loss_reg.item()) * bs
            total_cls += float(loss_cls.item()) * bs
            n += bs

            for i in range(bs):
                rows.append({
                    'record_id': int(meta_item['record_id'][i]),
                    'image_path': meta_item['image_path'][i],
                    'split': meta_item['split'][i],
                    'hgb_true': float(true_hgb[i]),
                    'hgb_pred_image': float(pred_hgb[i]),
                    'anemia_true': int(true_cls[i]),
                    'anemia_prob_image': float(prob[i]),
                    'sex_clean': meta_item['sex_clean'][i],
                    'race_group': meta_item['race_group'][i],
                    'ethnicity_clean': meta_item['ethnicity_clean'][i],
                })
    pred_df = pd.DataFrame(rows)
    losses = {'loss': total_loss / max(n,1), 'reg_loss': total_reg / max(n,1), 'cls_loss': total_cls / max(n,1)}
    return pred_df, losses


def aggregate_to_record(pred_df):
    rec = pred_df.groupby('record_id', as_index=False).agg({
        'hgb_true': 'first',
        'hgb_pred_image': 'mean',
        'anemia_true': 'first',
        'anemia_prob_image': 'mean',
        'sex_clean': 'first',
        'race_group': 'first',
        'ethnicity_clean': 'first',
        'split': 'first',
        'image_path': 'count',
    }).rename(columns={
        'hgb_pred_image': 'hgb_pred',
        'anemia_prob_image': 'anemia_prob',
        'image_path': 'n_images_used',
    })
    rec['anemia_pred'] = (rec['anemia_prob'] >= 0.5).astype(int)
    return rec


def compute_metrics(rec_pred):
    y = rec_pred['hgb_true'].values
    yp = rec_pred['hgb_pred'].values
    yc = rec_pred['anemia_true'].values.astype(int)
    pc = rec_pred['anemia_pred'].values.astype(int)
    prob = rec_pred['anemia_prob'].values

    out = {}
    out['n_records'] = int(len(rec_pred))
    out['mae'] = float(mean_absolute_error(y, yp))
    out['rmse'] = float(mean_squared_error(y, yp, squared=False))
    out['r2'] = float(r2_score(y, yp)) if len(np.unique(y)) > 1 else np.nan
    try:
        out['pearson_r'] = float(pearsonr(y, yp)[0])
    except Exception:
        out['pearson_r'] = np.nan
    try:
        out['spearman_r'] = float(spearmanr(y, yp).correlation)
    except Exception:
        out['spearman_r'] = np.nan
    out['accuracy'] = float(accuracy_score(yc, pc))
    out['balanced_accuracy'] = float(balanced_accuracy_score(yc, pc))
    pr, rc, f1, _ = precision_recall_fscore_support(yc, pc, average='binary', zero_division=0)
    out['precision'] = float(pr)
    out['recall'] = float(rc)
    out['f1'] = float(f1)
    if len(np.unique(yc)) == 2:
        out['roc_auc'] = float(roc_auc_score(yc, prob))
        out['average_precision'] = float(average_precision_score(yc, prob))
    else:
        out['roc_auc'] = np.nan
        out['average_precision'] = np.nan
    return out


BCE pos_weight: 0.7433264887063655


## 12. Train the model

Best checkpoint is selected by validation record-level MAE.


In [ ]:

history = []
best_val_mae = float('inf')
best_epoch = -1
bad_epochs = 0
best_ckpt = OUT_DIR / 'best_regnet400mf_fusion_no_leakage.pt'

for epoch in range(1, EPOCHS + 1):
    if epoch == FREEZE_BACKBONE_EPOCHS + 1:
        print('Unfreezing backbone...')
        set_backbone_trainable(model, True)
        optimizer = make_optimizer(model, backbone_trainable=True)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=2, factor=0.5)

    train_losses = train_one_epoch(model, train_loader, optimizer)
    val_pred_img, val_losses = collect_predictions(model, val_loader)
    val_pred_rec = aggregate_to_record(val_pred_img)
    val_metrics = compute_metrics(val_pred_rec)
    scheduler.step(val_metrics['mae'])

    row = {
        'epoch': epoch,
        **{f'train_{k}': v for k, v in train_losses.items()},
        **{f'val_{k}': v for k, v in val_losses.items()},
        **{f'val_metric_{k}': v for k, v in val_metrics.items()},
        'backbone_trainable': epoch > FREEZE_BACKBONE_EPOCHS,
    }
    history.append(row)

    print(
        f"Epoch {epoch:03d} | "
        f"train_loss={train_losses['loss']:.4f} | "
        f"val_loss={val_losses['loss']:.4f} | "
        f"val_MAE={val_metrics['mae']:.3f} | "
        f"val_RMSE={val_metrics['rmse']:.3f} | "
        f"val_F1={val_metrics['f1']:.3f}"
    )

    if val_metrics['mae'] < best_val_mae:
        best_val_mae = val_metrics['mae']
        best_epoch = epoch
        bad_epochs = 0
        torch.save({
            'model_state_dict': model.state_dict(),
            'epoch': epoch,
            'best_val_mae': best_val_mae,
            'demo_feature_cols': demo_feature_cols,
            'preprocess_info': preprocess_info,
            'config': {
                'image_size': IMAGE_SIZE,
                'include_race_in_model': INCLUDE_RACE_IN_MODEL,
                'include_ethnicity_in_model': INCLUDE_ETHNICITY_IN_MODEL,
            }
        }, best_ckpt)
        print('  saved best checkpoint:', best_ckpt)
    else:
        bad_epochs += 1
        if bad_epochs >= PATIENCE:
            print(f'Early stopping at epoch {epoch}. Best epoch: {best_epoch}, best val MAE: {best_val_mae:.3f}')
            break

history_df = pd.DataFrame(history)
history_df.to_csv(OUT_DIR / 'training_history.csv', index=False)
print('Saved:', OUT_DIR / 'training_history.csv')


/opt/anaconda3/envs/anemia_regnet/lib/python3.10/site-packages/torch/utils/data/dataloader.py:1095: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "/opt/anaconda3/envs/anemia_regnet/lib/python3.10/multiprocessing/spawn.py", line 116, in spawn_main
    exitcode = _main(fd, parent_sentinel)
  File "/opt/anaconda3/envs/anemia_regnet/lib/python3.10/multiprocessing/spawn.py", line 126, in _main
    self = reduction.pickle.load(from_parent)
AttributeError: Can't get attribute 'NailAnemiaDataset' on <module '__main__' (built-in)>


## 13. Plot training curves


In [ ]:

if 'history_df' not in globals() or len(history_df) == 0:
    history_df = pd.read_csv(OUT_DIR / 'training_history.csv')

plt.figure(figsize=(8, 5))
plt.plot(history_df['epoch'], history_df['train_loss'], label='train loss')
plt.plot(history_df['epoch'], history_df['val_loss'], label='val loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and validation loss')
plt.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / 'training_validation_loss.png', dpi=200)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(history_df['epoch'], history_df['val_metric_mae'], label='val MAE')
plt.plot(history_df['epoch'], history_df['val_metric_rmse'], label='val RMSE')
plt.xlabel('Epoch')
plt.ylabel('HGB error')
plt.title('Validation HGB regression error')
plt.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / 'validation_regression_error.png', dpi=200)
plt.show()

print('Saved figures to:', OUT_DIR)


## 14. Final evaluation on validation and test set

The checkpoint selected by validation MAE is loaded before final evaluation.


In [ ]:

ckpt = torch.load(best_ckpt, map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
model.to(device)
model.eval()
print('Loaded best epoch:', ckpt['epoch'], 'best val MAE:', ckpt['best_val_mae'])

val_pred_img, val_losses = collect_predictions(model, val_loader)
test_pred_img, test_losses = collect_predictions(model, test_loader)
val_pred_rec = aggregate_to_record(val_pred_img)
test_pred_rec = aggregate_to_record(test_pred_img)

val_metrics = compute_metrics(val_pred_rec)
test_metrics = compute_metrics(test_pred_rec)

metrics_df = pd.DataFrame([
    {'split': 'val', **val_metrics},
    {'split': 'test', **test_metrics},
])
metrics_df.to_csv(OUT_DIR / 'final_record_level_metrics.csv', index=False)
val_pred_img.to_csv(OUT_DIR / 'val_image_level_predictions.csv', index=False)
test_pred_img.to_csv(OUT_DIR / 'test_image_level_predictions.csv', index=False)
val_pred_rec.to_csv(OUT_DIR / 'val_record_level_predictions.csv', index=False)
test_pred_rec.to_csv(OUT_DIR / 'test_record_level_predictions.csv', index=False)

print('Final record-level metrics:')
display(metrics_df)
print('Saved final predictions and metrics to:', OUT_DIR)


## 15. Performance illustrations

This cell produces the main visual evidence:

- True vs predicted HGB plot
- Residual histogram
- Confusion matrix
- ROC curve
- Precision-recall curve


In [ ]:

# True vs predicted HGB.
plt.figure(figsize=(6, 6))
plt.scatter(test_pred_rec['hgb_true'], test_pred_rec['hgb_pred'], alpha=0.75)
lo = min(test_pred_rec['hgb_true'].min(), test_pred_rec['hgb_pred'].min())
hi = max(test_pred_rec['hgb_true'].max(), test_pred_rec['hgb_pred'].max())
plt.plot([lo, hi], [lo, hi], linestyle='--')
plt.xlabel('True HGB')
plt.ylabel('Predicted HGB')
plt.title('Test set: true vs predicted HGB')
plt.tight_layout()
plt.savefig(OUT_DIR / 'test_true_vs_pred_hgb.png', dpi=200)
plt.show()

# Residual histogram.
residual = test_pred_rec['hgb_pred'] - test_pred_rec['hgb_true']
plt.figure(figsize=(8, 5))
plt.hist(residual, bins=20)
plt.xlabel('Prediction error: predicted - true HGB')
plt.ylabel('Count')
plt.title('Test set HGB residuals')
plt.tight_layout()
plt.savefig(OUT_DIR / 'test_hgb_residual_histogram.png', dpi=200)
plt.show()

# Confusion matrix.
cm = confusion_matrix(test_pred_rec['anemia_true'], test_pred_rec['anemia_pred'], labels=[0, 1])
plt.figure(figsize=(5, 4))
plt.imshow(cm)
plt.xticks([0, 1], ['Pred non-anemia', 'Pred anemia'], rotation=20)
plt.yticks([0, 1], ['True non-anemia', 'True anemia'])
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, str(cm[i, j]), ha='center', va='center')
plt.title('Test set confusion matrix')
plt.tight_layout()
plt.savefig(OUT_DIR / 'test_confusion_matrix.png', dpi=200)
plt.show()

# ROC and PR curves.
if test_pred_rec['anemia_true'].nunique() == 2:
    fpr, tpr, _ = roc_curve(test_pred_rec['anemia_true'], test_pred_rec['anemia_prob'])
    plt.figure(figsize=(6, 5))
    plt.plot(fpr, tpr, label=f"AUC={test_metrics['roc_auc']:.3f}")
    plt.plot([0, 1], [0, 1], linestyle='--')
    plt.xlabel('False positive rate')
    plt.ylabel('True positive rate')
    plt.title('Test set ROC curve')
    plt.legend()
    plt.tight_layout()
    plt.savefig(OUT_DIR / 'test_roc_curve.png', dpi=200)
    plt.show()

    prec, rec, _ = precision_recall_curve(test_pred_rec['anemia_true'], test_pred_rec['anemia_prob'])
    plt.figure(figsize=(6, 5))
    plt.plot(rec, prec, label=f"AP={test_metrics['average_precision']:.3f}")
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.title('Test set precision-recall curve')
    plt.legend()
    plt.tight_layout()
    plt.savefig(OUT_DIR / 'test_precision_recall_curve.png', dpi=200)
    plt.show()
else:
    print('ROC/PR skipped because the test set has only one class.')

print('Saved performance figures to:', OUT_DIR)


## 16. Subgroup / fairness-style analysis

This does not prove fairness, but it shows whether errors differ strongly across demographic groups.

Reports are computed on record-level predictions, not image-level predictions.


In [ ]:

def subgroup_metrics(df, group_col, min_n=5):
    rows = []
    for group, g in df.groupby(group_col):
        if len(g) < min_n:
            continue
        m = compute_metrics(g)
        rows.append({'group_col': group_col, 'group': group, **m})
    return pd.DataFrame(rows)

fairness_parts = []
for col in ['sex_clean', 'race_group', 'ethnicity_clean']:
    fairness_parts.append(subgroup_metrics(test_pred_rec, col, min_n=5))
fairness_df = pd.concat(fairness_parts, ignore_index=True) if fairness_parts else pd.DataFrame()
fairness_df.to_csv(OUT_DIR / 'test_subgroup_metrics.csv', index=False)
print('Saved:', OUT_DIR / 'test_subgroup_metrics.csv')
display(fairness_df)

# Simple bar plot of MAE by subgroup.
if len(fairness_df):
    plot_df = fairness_df.copy()
    plot_df['label'] = plot_df['group_col'] + ': ' + plot_df['group'].astype(str)
    plot_df = plot_df.sort_values('mae')
    plt.figure(figsize=(10, max(4, 0.35 * len(plot_df))))
    plt.barh(plot_df['label'], plot_df['mae'])
    plt.xlabel('MAE')
    plt.title('Test set HGB MAE by subgroup')
    plt.tight_layout()
    plt.savefig(OUT_DIR / 'test_subgroup_mae.png', dpi=200)
    plt.show()


## 17. Save a compact model card / run report


In [ ]:

run_report = {
    'task': 'Anemia prediction from preprocessed nail images',
    'model': 'RegNet-Y-400MF + demographics intermediate fusion + regression/classification heads',
    'no_ita_skin_tone_input': True,
    'data_split': {
        'unit': 'record_id / patient',
        'test_size': TEST_SIZE,
        'val_size': VAL_SIZE,
        'patient_overlap_check_passed': True,
        'hash_leakage_check_run': RUN_HASH_LEAKAGE_CHECK,
    },
    'csv_shapes': {
        'FileDirectory.csv': list(filedir_raw.shape),
        'PredictingAnemiaInCo-CJStudyData_DATA_2023-07-12_0653.csv': list(lab_raw.shape),
        'Race and gender file 070425.csv': list(demo_raw.shape),
    },
    'dataset_after_cleaning': {
        'records_with_matched_images': int(image_df_fe['record_id'].nunique()),
        'image_rows_with_matched_paths': int(len(image_df_fe)),
        'train_records': int(train_df['record_id'].nunique()),
        'val_records': int(val_df['record_id'].nunique()),
        'test_records': int(test_df['record_id'].nunique()),
        'train_images': int(len(train_df)),
        'val_images': int(len(val_df)),
        'test_images': int(len(test_df)),
    },
    'preprocessing': preprocess_info,
    'best_epoch': int(ckpt['epoch']),
    'final_metrics': metrics_df.to_dict(orient='records'),
}
with open(OUT_DIR / 'run_report.json', 'w') as f:
    json.dump(run_report, f, indent=2)
print(json.dumps(run_report, indent=2)[:3000])
print('Saved:', OUT_DIR / 'run_report.json')


## 18. Inference helper for new images

Use this after training to predict HGB/anemia for new preprocessed nail image(s).

For multiple nail crops from the same person, pass all image paths and the function will average predictions.


In [ ]:

def build_demo_vector_for_one(age, sex='Not Sure', race_group='Unknown', ethnicity_clean='Unknown'):
    row = pd.DataFrame([{
        'age_num': age,
        'sex_clean': normalize_sex(sex),
        'race_group': clean_text(race_group),
        'ethnicity_clean': clean_text(ethnicity_clean),
    }])
    # Match expected columns used by the encoder.
    for col in cat_cols:
        if col not in row.columns:
            row[col] = 'Unknown'
    age_series = pd.to_numeric(row['age_num'], errors='coerce').fillna(age_median).astype(float)
    row['age_z'] = ((age_series - age_mean) / age_std).round(1)
    cat = ohe.transform(row[cat_cols].fillna('Unknown'))
    try:
        cat_names = ohe.get_feature_names_out(cat_cols).tolist()
    except Exception:
        cat_names = [f'cat_{i}' for i in range(cat.shape[1])]
    cat_df = pd.DataFrame(cat, columns=cat_names, index=row.index)
    row = pd.concat([row, cat_df], axis=1)
    return torch.tensor(row[demo_feature_cols].astype(float).values, dtype=torch.float32)


def predict_person_images(image_paths, age, sex='Not Sure', race_group='Unknown', ethnicity_clean='Unknown'):
    model.eval()
    demo = build_demo_vector_for_one(age, sex, race_group, ethnicity_clean).to(device)
    preds_hgb = []
    preds_prob = []
    with torch.no_grad():
        for p in image_paths:
            img = Image.open(p).convert('RGB')
            img = eval_transform(img).unsqueeze(0).to(device)
            pred_z, logit = model(img, demo)
            pred_hgb = float(pred_z.cpu().numpy().reshape(-1)[0] * hgb_std + hgb_mean)
            prob = float(torch.sigmoid(logit).cpu().numpy().reshape(-1)[0])
            preds_hgb.append(pred_hgb)
            preds_prob.append(prob)
    return {
        'hgb_pred_mean': float(np.mean(preds_hgb)),
        'hgb_pred_per_image': preds_hgb,
        'anemia_prob_mean': float(np.mean(preds_prob)),
        'anemia_pred': int(np.mean(preds_prob) >= 0.5),
    }

# Example after training:
# result = predict_person_images(['/path/to/nail1.png', '/path/to/nail2.png'], age=33, sex='M')
# result


# Done

After running all cells, the important outputs are in `OUT_DIR`:

- `best_regnet400mf_fusion_no_leakage.pt`
- `final_record_level_metrics.csv`
- `test_record_level_predictions.csv`
- `test_true_vs_pred_hgb.png`
- `test_confusion_matrix.png`
- `test_roc_curve.png`
- `test_subgroup_metrics.csv`
- `run_report.json`

The notebook stops if patient-level leakage or identical-image hash leakage is detected.
